# Final Project: Adaptive Feature Selection for Predicting Video Quality Degradation

**Machine Learning for Computer Systems** · Open Problem / Research  
**Aeliya Grover, Clarisse Cheung** with Assistance from Cursor: Composer 2.5, GPT 5.6 - Sol 

---

## What This Notebook Is For

This notebook is the **primary deliverable** for the course final project. It should run end-to-end from setup through evaluation and conclusions. Your companion **Sphinx project report** will present the same work in a portfolio-quality writeup; use this notebook as the source of truth for code, experiments, and results.

## Research Question

Can *adaptive feature selection*—starting with cheap features and computing expensive ones only when needed—reduce the computational cost of predicting video quality degradation while matching the performance of a model that always uses the full feature set?

## How This Notebook Is Organized

Each part below explains **what you will do** and **why it matters** for the larger project goal. Work through the parts in order; later steps depend on earlier decisions (splits, labels, feature tiers, and baselines).

| Part | What | Why it matters |
|------|------|----------------|
| 1 | Setup & configuration | Reproducibility and one-click execution |
| 2 | Data loading & integrity | Trustworthy inputs before any modeling |
| 3 | Labels, splits & leakage checks | Correct task definition and honest evaluation |
| 4 | Feature tiers (L3 / L4 / L7) | Connect ML inputs to systems cost |
| 5 | Baselines & controls | Know what "good" looks like before the adaptive method |
| 6 | Train multiple models | Compare fixed tiers, full features, and resolution-only |
| 7 | Adaptive cascade | Main research contribution |
| 8 | Hyperparameter tuning | Fair comparison across models |
| 9 | Prediction evaluation | Measure quality with imbalance-aware metrics |
| 10 | Systems-cost evaluation | Measure the deployment tradeoff |
| 11 | Error analysis & robustness | Understand failures, not just averages |
| 12 | Cross-model comparison | Synthesize results across all experiments |
| 13 | Conclusions | Answer the research question and reflect on learning |
| 14 | Reproducibility checklist | Verify the notebook is submission-ready |

> **Note:** This is a blank template. Fill in code, results, plots, and written responses as you complete the project. Do not skip the written justification cells—they are part of your grade.


---

## Part 1: Setup & Configuration

**What:** Install dependencies, set random seeds, and define paths to data files.

**Why:** The project must run end-to-end with "Restart kernel and run all." Centralizing configuration here makes the rest of the notebook reproducible and easier to debug.

### 1.1 Clone or locate the data

You will need:
- `video_dataset.pkl` — 204,713 ten-second windows across 4,000 sessions (Netflix, YouTube, Twitch, Amazon Prime Video)
- `netflix.pcap` — packet capture for empirical feature-extraction cost measurement

If you have not already cloned the course data repository:

```bash
git clone https://github.com/noise-courses/data.git
```

Point the paths below to wherever these files live on your machine.


In [ ]:
# TODO: imports, random seeds, and path configuration
# Example paths (update as needed):
# VIDEO_DATASET_PATH = ".../video_dataset.pkl"
# NETFLIX_PCAP_PATH = ".../netflix.pcap"
# RANDOM_STATE = 42
# HORIZON_SECONDS = ...  # sweep 10, 20, 30


**Document your environment.** List the Python version and key library versions you used. This helps anyone reproducing your work.


---

## Part 2: Data Loading & Integrity Checks

**What:** Load `video_dataset.pkl`, inspect its schema, and verify basic properties (row counts, services, resolution values, missing data).

**Why:** Assignment 1 showed that bad labels and invalid resolutions silently hurt models. Catching data issues early prevents wasted training runs and misleading results.


In [ ]:
# TODO: load video_dataset.pkl and inspect shape, columns, dtypes


### 2.1 Exploratory summary

Summarize:
- Number of sessions, windows, and services
- Valid resolution values (240, 360, 480, 720, 1080)
- Distribution of services (Netflix, YouTube, Twitch, Amazon)
- Any columns with missing or suspicious values


In [ ]:
# TODO: EDA — session counts, service breakdown, resolution distribution


**Why did you keep or remove any rows/columns?** *(Write your answer below.)*


---

## Part 3: Labels, Splits & Leakage Checks

**What:** Construct the **future-horizon binary label**: at time *t*, predict whether resolution drops during *(t, t+Δ]*. Split data by **session** (no session in both train and test) and optionally by **time** (train on earlier sessions, test on later ones for drift analysis).

**Why:** This is the core prediction task. Session-level splits prevent leakage across windows from the same viewing session. Temporal splits test whether the model holds up as network conditions change.


### 3.1 Define the prediction horizon Δ

You proposed sweeping Δ from 10 to 30 seconds. Pick your primary horizon and note any additional horizons you will compare.


In [ ]:
# TODO: construct future-horizon downswitch labels for your chosen Δ


### 3.2 Class balance

Positive cases are rare (~1.7–3.9% of windows). Report the positive rate overall and per service.

**Why does class imbalance matter for metric choice?** *(Write your answer below.)*


In [ ]:
# TODO: report label distribution overall and per service


### 3.3 Train / validation / test splits

- **Session split:** no session appears in more than one split
- **Temporal split (optional):** train on earlier sessions, test on later ones


In [ ]:
# TODO: create session-level (and optional temporal) splits


### 3.4 Leakage checks

Identify any features or columns that would leak future information (e.g., resolution at t+Δ, post-event statistics). Remove or exclude them from training.

**List the columns you excluded and why.** *(Write your answer below.)*


In [ ]:
# TODO: identify and remove leaky columns


---

## Part 4: Feature Tiers (L3 / L4 / L7)

**What:** Group features by protocol layer—the cost hierarchy from your proposal:

| Tier | Count | Examples | Systems cost |
|------|-------|----------|--------------|
| **L3** | 11 | throughput, byte/packet counts, parallel flows | Cheap — packet headers only |
| **L4** | 95 | RTT, bytes in flight, retransmissions, receive window | Medium — per-flow TCP state |
| **L7** | 55 | chunk sizes, chunk inter-arrival times | Expensive — infer app structure in encrypted traffic |

**Why:** The entire project compares *prediction quality* against *feature cost*. You cannot evaluate the adaptive cascade without explicitly defining which features belong to each tier.


In [ ]:
# TODO: define feature column lists for L3, L4, L7, and full (L3+L4+L7)


**Briefly justify why these tier groupings match the systems-cost hierarchy.** *(Write your answer below.)*


---

## Part 5: Baselines & Control Experiment Design

**What:** Before building the adaptive cascade, define the comparison points:

1. **Resolution-only baseline** — current resolution alone (sessions at lowest resolution cannot downswitch)
2. **Fixed-tier models** — L3-only, L4-only, L7-only, and full feature set
3. **Random escalation control** — escalate to higher tiers at random, matched to the cascade's escalation rate

**Why:** Without these controls, you cannot tell whether the cascade wins because adaptation is smart, or simply because more features sometimes help. Every result should be reported as improvement over the resolution-only baseline.


**Describe your experimental matrix.** Which models × feature sets × splits will you train? *(Write your plan below.)*


---

## Part 6: Train Multiple Models

**What:** Train and save models for each baseline and fixed-tier configuration. Leave separate sections so you can compare them fairly in Part 12.

**Why:** The project requires comparing multiple approaches—not just one model. Document hyperparameters and training time for each.


### 6.1 Resolution-only baseline

Train a classifier using **only the current resolution** as input.


In [ ]:
# TODO: train resolution-only baseline


### 6.2 Model A — Random Forest (full features)

Train a random forest on the **full feature set** (L3 + L4 + L7).


In [ ]:
# TODO: train Random Forest on full features


### 6.3 Model B — Gradient Boosting (full features)

Train a gradient boosting classifier on the **full feature set**.


In [ ]:
# TODO: train Gradient Boosting on full features


### 6.4 Fixed-tier models

Train separate classifiers on **L3-only**, **L4-only**, and **L7-only** feature sets. Use the same model family and comparable hyperparameter search for each.


In [ ]:
# TODO: train L3-only model


In [ ]:
# TODO: train L4-only model


In [ ]:
# TODO: train L7-only model


**Record training time and key hyperparameters for each model above.** *(Write your notes below.)*


---

## Part 7: Adaptive Cascade

**What:** Implement sequential feature acquisition:

1. Classify with **L3 features** first; output a calibrated probability
2. If probability is near the decision boundary (model is unsure), compute **L4 features** and reclassify
3. If still unsure, compute **L7 features** and reclassify
4. Stop when confidence is high enough or the full feature set is reached

**Why:** This is the main research contribution—spending compute only on hard examples while aiming to match full-feature performance.


### 7.1 Calibrate probabilities

Because cascade thresholds operate on probabilities, calibrate each tier's classifier (e.g., Platt scaling or isotonic regression) so outputs reflect true likelihoods.


In [ ]:
# TODO: calibrate tier classifiers


### 7.2 Implement the cascade logic

Define confidence thresholds and escalation rules.


In [ ]:
# TODO: implement adaptive cascade


### 7.3 Random escalation control

Implement random tier escalation at the **same average escalation rate** as the cascade, for fair comparison.


In [ ]:
# TODO: implement random escalation control


**Explain your threshold choices and why calibration matters for the cascade.** *(Write your answer below.)*


---

## Part 8: Hyperparameter Tuning & Validation

**What:** Tune hyperparameters for each model using cross-validation on the training set. Use the validation split (or CV) to select settings—do not peek at the test set.

**Why:** Fair comparison requires that every model gets a comparable tuning effort. Document your search space and best parameters.


In [ ]:
# TODO: hyperparameter tuning (e.g., GridSearchCV or RandomizedSearchCV)


**Summarize best hyperparameters per model.** *(Write your table or notes below.)*


---

## Part 9: Prediction Evaluation

**What:** Evaluate all models on the **held-out test set** using imbalance-aware metrics:

- **PR-AUC** (headline metric)
- Precision, recall, F1
- Confusion matrix
- Results **per service** (pooled and broken out)
- **Lead time** — seconds before a downswitch the model first flags the session

**Why:** Downswitches are rare (<4% of windows). Accuracy and ROC-AUC can look strong even for useless models. PR-AUC and per-service breakdowns reveal whether the model actually helps.


In [ ]:
# TODO: compute PR-AUC, precision, recall, F1 for all models


In [ ]:
# TODO: plot confusion matrices


In [ ]:
# TODO: per-service evaluation breakdown


In [ ]:
# TODO: compute lead time for flagged sessions


**Interpret the prediction results.** Which models perform best? Does the cascade match full-feature performance? *(Write your analysis below.)*


---

## Part 10: Systems-Cost Evaluation

**What:** Measure the deployment tradeoff between prediction quality and computational cost:

1. **Empirical feature-extraction time** for each tier on `netflix.pcap`
2. **Quality–cost curve** — sweep cascade confidence thresholds; plot PR-AUC vs. average feature cost
3. **Operating points** — settings that stay close to full-model performance at substantially lower cost
4. **Time-to-availability model** — L7 features cannot exist until chunks arrive; plot quality vs. effective warning time (label clearly as a *model*, not a direct measurement)
5. Compare cascade vs. fixed tiers vs. **random escalation** at matched rate

**Why:** The research question is about *systems cost*, not accuracy alone. This part answers whether adaptive selection is worth deploying.


In [ ]:
# TODO: measure feature-extraction time per tier on netflix.pcap


In [ ]:
# TODO: sweep cascade thresholds and plot quality vs. cost


In [ ]:
# TODO: identify operating points on the quality-cost curve


In [ ]:
# TODO: model time-to-availability for L7 features


In [ ]:
# TODO: compare cascade vs. random escalation at matched rate


**Interpret the cost results.** At what operating point does the cascade offer the best tradeoff? Does any fixed tier dominate on both axes? *(Write your analysis below.)*


---

## Part 11: Error Analysis & Robustness Checks

**What:** Go beyond aggregate metrics. Investigate:

- **False positives** — windows flagged but no downswitch occurred
- **False negatives** — missed downswitches
- **Calibration quality** — do predicted probabilities match observed rates?
- **Temporal drift** — does performance drop on the temporal test split?
- **Failure cases** — sessions or services where the model consistently fails
- **Edge cases** — sessions already at lowest resolution, very short sessions, etc.

**Why:** A model that looks good on average may fail in deployment-critical situations. Error analysis builds confidence (or reveals problems) before you draw conclusions.


In [ ]:
# TODO: analyze false positives and false negatives


In [ ]:
# TODO: calibration plots (reliability diagram)


In [ ]:
# TODO: compare session-split vs. temporal-split performance


**What failure modes did you find? What would you do differently?** *(Write your answer below.)*


---

## Part 12: Cross-Model Comparison & Interpretation

**What:** Synthesize results from Parts 6–11 into a single comparison table and visualization set. Compare:

| Model | Feature set | PR-AUC | Avg. feature cost | Lead time | Notes |
|-------|-------------|--------|-------------------|-----------|-------|
| Resolution-only | resolution | | | | |
| L3-only | L3 | | | | |
| L4-only | L4 | | | | |
| L7-only | L7 | | | | |
| RF full | L3+L4+L7 | | | | |
| GB full | L3+L4+L7 | | | | |
| Cascade | adaptive | | | | |
| Random escalation | matched | | | | |

**Why:** The conclusion depends on seeing all experiments together—not isolated best numbers.


In [ ]:
# TODO: build comparison table and summary plots


**Summarize the key findings across all models.** *(Write your interpretation below.)*


---

## Part 13: Conclusions

**What:** Answer the research question and reflect on what you learned. This section should be written in plain language, suitable for a portfolio or job interview.

**Why:** Results without conclusions do not complete the project. Tie your findings back to the learning objective: how feature representation and selection affect both model quality and systems cost.


### 13.1 Research question

*Can adaptive feature selection reduce computational cost while matching full-feature performance?*

**Your answer:** *(Write below.)*


### 13.2 Key findings

*(Bullet the 3–5 most important results.)*


### 13.3 Limitations

*(What could not you answer with this data or setup?)*


### 13.4 Future work

*(What would you try next if you had more time?)*


### 13.5 Learning objective reflection

*We expected to learn how feature representation and selection affect both model quality and systems cost, and to practice designing an adaptive pipeline that spends compute only on hard examples.*

**What did you learn?** *(Write below.)*


---

## Part 14: Reproducibility & Submission Checklist

Before submitting, verify each item:

- [ ] **Restart kernel and run all** — notebook executes top-to-bottom without errors
- [ ] All data paths are documented (or data is included / linked)
- [ ] Random seeds are set for reproducibility
- [ ] Every plot and table has a brief caption or explanation
- [ ] Written justification cells are filled in (not left blank)
- [ ] Comparison table (Part 12) is complete
- [ ] Conclusions (Part 13) directly answer the research question
- [ ] Companion **Sphinx report** is prepared with inline code excerpts
- [ ] Group responsibilities are documented (if working in a group of 2–3)

**Final sign-off:** *(Note any known issues or assumptions for the grader.)*
